# Teste isolado — ANP (Notícias e Comunicados)

Fonte candidata: **ANP - Agência Nacional do Petróleo, Gás Natural e
Biocombustíveis**, setor Energia. Notebook **descartável** (Fase 1) — sem
dispatcher, sem `atualizar_status_fonte`, sem gravar nada. Só valida:

1. Scraping da listagem (categoria, título, data, link)
2. Extração do texto completo de uma notícia individual

## Confirmado antes de assumir: mesma plataforma da ANA

Verifiquei a listagem antes de escrever qualquer código — é a mesma
plataforma Plone/gov.br já validada com ANA (e ANTT/ANEEL no dispatcher
genérico): HTML renderizado no servidor, sem JavaScript, mesma estrutura
`ul.noticias.listagem-noticias-com-foto`, mesma paginação `?b_start:int=N`.
Nenhuma URL temporária aqui (diferente da ANA, que tem uma URL de período
eleitoral) — este é o endereço padrão/permanente de notícias da ANP.

`robots.txt` de `gov.br/anp` não bloqueia (`Disallow:` vazio pra `User-agent: *`).

## Sem filtro de relevância nesta etapa

Confirmado na amostra: a listagem mistura ruído administrativo (ex.:
"Fechamento antecipado do protocolo", "Reunião de Diretoria...") com
conteúdo regulatório relevante (leilões, resoluções, consulta pública sobre
gas release, dados de produção). Captura tudo — filtragem de relevância é
responsabilidade da etapa de NLP mais adiante no pipeline, não da captura.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
import unicodedata
import urllib.parse
from datetime import datetime
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://www.gov.br/anp/pt-br/canais_atendimento/imprensa/noticias-comunicados"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form", "nav", "header", "footer", "aside", "button"]
SELETORES_CONTEUDO = ["#content-core", "#parent-fieldname-text", "article", "main"]
PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
PADRAO_DATA_PUBLICADO = re.compile(r"Publicado em\s*(\d{2}/\d{2}/\d{4})")
ITENS_POR_PAGINA = 30

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

Mesmo padrão de `listar_ana` (já em produção no dispatcher genérico
`ingest-scraping`): `ul.noticias.listagem-noticias-com-foto > li`,
paginação via `?b_start:int=N`. A ANP tem histórico bem maior que a ANA (a
barra de paginação mostra ~15 páginas) — limito a amostra aqui pra não
varrer o histórico inteiro só pra testar.

In [0]:
def listar_anp(max_paginas: int = 3, itens_por_pagina: int = ITENS_POR_PAGINA) -> list[dict]:
    itens = []

    for pagina in range(max_paginas):
        b_start = pagina * itens_por_pagina
        url_pagina = SITE_URL if b_start == 0 else f"{SITE_URL.rstrip('/')}?b_start:int={b_start}"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página (b_start={b_start}); parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        lista = soup.select_one("ul.noticias.listagem-noticias-com-foto")
        if not lista:
            print(f"  -> nenhuma lista encontrada na página (b_start={b_start}); parando.")
            break

        lis = lista.find_all("li", recursive=False)
        if not lis:
            print(f"  -> página vazia (b_start={b_start}); fim da listagem.")
            break

        for li in lis:
            tag_a = li.select_one("h2.titulo a")
            if not tag_a:
                continue
            categoria = li.select_one("div.categoria-noticia")
            data = li.select_one("span.data")

            data_publicacao = None
            if data:
                texto_data = data.get_text(strip=True)
                m = re.match(r"(\d{2})/(\d{2})/(\d{4})", texto_data)
                if m:
                    dia, mes, ano = m.groups()
                    data_publicacao = f"{ano}-{mes}-{dia}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": tag_a["href"].strip(),
                "categoria": categoria.get_text(strip=True) if categoria else None,
                "published_at": data_publicacao,
            })

        print(f"  página b_start={b_start}: {len(lis)} itens.")
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_anp()

print(f"\n{len(itens)} notícias listadas.\n")
print(f"{'DATA':<12} {'CATEGORIA':<32} TÍTULO")
print("-" * 110)
for item in itens:
    categoria = (item["categoria"] or "?")[:30]
    print(f"{item['published_at'] or '?':<12} {categoria:<32} {item['titulo'][:55]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]
sem_categoria = [i for i in itens if not i["categoria"]]
categorias_distintas = sorted({i["categoria"] for i in itens if i["categoria"]})

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)} | sem categoria: {len(sem_categoria)}")
print(f"categorias distintas na amostra: {categorias_distintas}")
print(f"\nExemplo de link: {itens[0]['url']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

Mesmos seletores já usados em `extrair_texto_generico` do dispatcher
`ingest-scraping` e no teste da ANA (`#content-core` / `#parent-fieldname-text`).
Amostra inclui de propósito um item administrativo curto (ex.: aviso de
fechamento de protocolo) e itens mais substanciais, pra confirmar que o
`MIN_CHARS_TEXTO` do dispatcher genérico realmente separa ruído de conteúdo
relevante em vez de descartar tudo.

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    base = None
    for seletor in SELETORES_CONTEUDO:
        encontrado = soup.select_one(seletor)
        if encontrado and len(encontrado.get_text(strip=True)) > 100:
            base = encontrado
            break
    if base is None:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def data_publicado_anp(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    m = PADRAO_DATA_PUBLICADO.search(soup.get_text(" ", strip=True))
    if not m:
        return None
    dia, mes, ano = m.group(1).split("/")
    return f"{ano}-{mes}-{dia}"


def extrair_noticia_anp(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)
    data_pagina = data_publicado_anp(html)

    return {
        "titulo": item["titulo"],
        "url": item["url"],
        "categoria": item["categoria"],
        "published_at": data_pagina or item["published_at"],
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_anp(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars (prováveis avisos administrativos): {len(curtas)}")
for d in curtas:
    print(f"  - ({len(d['texto'])} chars) {d['titulo'][:70]}")

In [0]:
# Amostra completa de uma notícia substancial — pra conferir na mão se bate
# com o que aparece no site. Pega a maior das capturadas.
detalhe = max(detalhes, key=lambda d: len(d["texto"]))

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"CATEGORIA   : {detalhe['categoria']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai categoria/título/data/link
de todos os itens, e o texto completo sai limpo dos mesmos seletores já em
produção. Confirma o que o pedido antecipava: itens administrativos curtos
(ex.: aviso de fechamento de protocolo) e itens substanciais (resoluções,
leilões, consultas públicas) convivem na mesma listagem, sem filtro — como
esperado, a triagem fica pra etapa de NLP.

**Avaliação preliminar para a Fase 2**: mesma conclusão da ANA — encaixa no
dispatcher genérico `ingest-scraping`, reaproveitando `listar_ana` (mesma
estrutura, só muda `SITE_URL`) ou uma função `listar_anp` equivalente,
`extrair_titulo_h1` e `data_govbr_plone` já existentes. Sem Selenium, sem
parsing particular.

**Pendências antes da Fase 3 (registro em produção)** — meu pedido explícito
foi confirmar isso antes de registrar, então deixo como próximo passo, não
resolvido aqui:
1. Confirmar que `source_id` escolhido (ex.: `"anp_noticias"`) não colide —
   já conferi no repo local (`grep` não achou nada), falta conferir contra
   `controle_fontes` ao vivo no Databricks antes do INSERT.
2. Definir `importancia_original` (Alta/Média/Baixa) — **não decido isso
   sozinho**, fica para confirmação explícita antes do registro.